# 🌐 Day 03a: Virtual Memory — Demand Paging, Page Faults & Thrashing

---

## 🎯 What You'll Master Today
- Virtual memory concept — running programs larger than RAM
- Demand paging & lazy loading
- Page fault handling sequence
- Copy-on-Write (COW)
- Thrashing — when paging kills performance

---

## 🏠 The House Renovation Analogy

Virtual memory = a small workshop but a huge warehouse:
- **Workshop** (RAM) = limited space to work
- **Warehouse** (Disk) = stores everything you own
- **Demand paging** = only bring tools from warehouse when you NEED them
- **Page fault** = "I need the drill!" → trip to warehouse
- **Thrashing** = spending more time going to warehouse than actually working

```
╔══════════════════════════════════════════════════════════╗
║                 VIRTUAL MEMORY FLOW                      ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  CPU → Virtual Addr → Page Table → Frame? ─YES→ Access  ║
║                                      │                   ║
║                                      NO (valid=0)        ║
║                                      ↓                   ║
║                                  PAGE FAULT              ║
║                                      ↓                   ║
║                              Trap to OS kernel           ║
║                                      ↓                   ║
║                          Find page on disk (swap)        ║
║                                      ↓                   ║
║                     Find free frame (or evict one)       ║
║                                      ↓                   ║
║                       Load page → Update table           ║
║                                      ↓                   ║
║                        Restart instruction               ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Demand Paging Simulator
# ============================================

class DemandPagingSimulator:
    def __init__(self, num_frames: int, total_pages: int):
        self.num_frames = num_frames
        self.frames = [None] * num_frames    # physical memory
        self.page_table = {}                  # page → frame mapping
        self.valid_bit = [False] * total_pages
        self.page_faults = 0
        self.accesses = 0
    
    def access_page(self, page: int) -> str:
        self.accesses += 1
        
        if page in self.page_table:
            return f"  Page {page} → Frame {self.page_table[page]} ✅ (hit)"
        
        # PAGE FAULT
        self.page_faults += 1
        
        # Find free frame
        free_frame = None
        for i, f in enumerate(self.frames):
            if f is None:
                free_frame = i
                break
        
        if free_frame is not None:
            self.frames[free_frame] = page
            self.page_table[page] = free_frame
            return f"  Page {page} → PAGE FAULT! Loaded to Frame {free_frame} 📥"
        else:
            return f"  Page {page} → PAGE FAULT! No free frames — need replacement 🔄"
    
    def stats(self):
        rate = (self.page_faults / self.accesses * 100) if self.accesses else 0
        print(f"\n  📊 Page Faults: {self.page_faults}/{self.accesses} ({rate:.1f}%)")
        print(f"  Memory: {[f'P{p}' if p is not None else '---' for p in self.frames]}")


# 4 frames of RAM, 8 total pages
sim = DemandPagingSimulator(num_frames=4, total_pages=8)

# Access pattern — first access of each page = fault
accesses = [0, 1, 2, 3, 0, 1, 4, 0, 1, 2]
print("  Demand Paging Simulation (4 frames)")
print("  " + "═" * 45)

for page in accesses:
    result = sim.access_page(page)
    print(result)

sim.stats()

In [ ]:
# ============================================
# 2. Copy-on-Write (COW) Simulation
# ============================================

import copy

class COWPage:
    """Simulates Copy-on-Write memory pages."""
    
    def __init__(self, data: list):
        self._data = data
        self._ref_count = 1
        self._id = id(data)
    
    def fork(self) -> 'COWPage':
        """Fork: share the same page, increment ref count."""
        child = COWPage.__new__(COWPage)
        child._data = self._data  # SHARED — no copy yet!
        child._ref_count = 1
        child._id = self._id
        self._ref_count += 1
        print(f"  fork(): Parent & child SHARE page (ref_count={self._ref_count})")
        print(f"  Same physical page: {id(self._data) == id(child._data)}")
        return child
    
    def write(self, index: int, value):
        """Write triggers copy if page is shared."""
        if self._ref_count > 1 or id(self._data) == self._id:
            print(f"  write(): COW triggered! Copying page before write...")
            self._data = copy.deepcopy(self._data)  # NOW we copy
            self._ref_count = 1
        self._data[index] = value
        print(f"  write(): Page[{index}] = {value}")


# Demo: fork() without COW wastes memory
print("  Copy-on-Write Demo")
print("  " + "═" * 40)

parent = COWPage(["code", "data", "stack", "heap"])
print(f"  Parent page: {parent._data}")
print()

# fork() — child shares pages
child = parent.fork()
print()

# Child writes — NOW COW kicks in
child.write(1, "MODIFIED_DATA")
print(f"\n  Parent: {parent._data}")
print(f"  Child:  {child._data}")
print(f"  Different physical pages: {id(parent._data) != id(child._data)} ✅")

In [ ]:
# ============================================
# 3. Thrashing Detection — Working Set Model
# ============================================

def working_set_analysis(reference_string: list[int], window_size: int, num_frames: int):
    """
    Analyze working set to detect potential thrashing.
    Working set = set of pages referenced in the last 'window_size' accesses.
    Thrashing = working set > available frames.
    """
    print(f"  Working Set Analysis (window={window_size}, frames={num_frames})")
    print("  " + "═" * 50)
    
    thrashing_points = []
    
    for i in range(len(reference_string)):
        start = max(0, i - window_size + 1)
        window = reference_string[start:i+1]
        ws = set(window)
        ws_size = len(ws)
        
        status = "⚠️ THRASHING" if ws_size > num_frames else "✅ OK"
        if ws_size > num_frames:
            thrashing_points.append(i)
        
        print(f"  t={i:2d} | Ref: P{reference_string[i]} | WS: {ws} | Size: {ws_size} | {status}")
    
    if thrashing_points:
        print(f"\n  ⚠️ Thrashing at {len(thrashing_points)} points! Need more frames or better locality.")
    else:
        print(f"\n  ✅ No thrashing — working set fits in {num_frames} frames.")


# Good locality → working set fits
print("  === Good Locality ===")
working_set_analysis([1, 2, 1, 2, 1, 3, 3, 2, 1, 2], window_size=4, num_frames=3)

print()

# Bad locality → thrashing!
print("  === Bad Locality (Thrashing) ===")
working_set_analysis([1, 2, 3, 4, 5, 1, 2, 3, 4, 5], window_size=4, num_frames=3)

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is demand paging? | Load pages only when accessed (lazy). Page fault triggers disk I/O |
| 2 | Steps in page fault handling? | Trap → check valid → find on disk → get frame → load → update table → restart |
| 3 | What is Copy-on-Write? | fork() shares pages; copy only when process writes. Saves memory on fork |
| 4 | What causes thrashing? | Working set > available frames. Process spends more time paging than executing |
| 5 | How to fix thrashing? | Increase RAM, reduce degree of multiprogramming, use working set model |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Virtual memory: programs can exceed physical RAM      │
## │  • Page fault = page not in RAM → load from disk (~10ms)│
## │  • COW: fork() shares pages, copy only on write          │
## │  • Thrashing: working set > frames → constant faulting   │
## │  • Locality of reference makes VM work efficiently       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Page Replacement** — FIFO, LRU, Optimal & Belady's anomaly